# 01 - Ingest

In [ ]:
# Databricks Notebook: 01-ingest
# Purpose: Ingest raw data into Bronze using Auto Loader

from pyspark.sql.functions import current_timestamp


In [ ]:

# ---------------------------
# CONFIGURATION
# ---------------------------
catalog = "main"
schema = "bronze"
table_name = "events_bronze"

raw_path = "/mnt/raw/events/"
checkpoint_path = "/mnt/checkpoints/events_bronze/"

full_table_name = f"{catalog}.{schema}.{table_name}"


In [ ]:

# ---------------------------
# INGEST RAW DATA WITH AUTO LOADER
# ---------------------------

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(raw_path)
        .withColumn("ingest_timestamp", current_timestamp())
)


In [ ]:

# ---------------------------
# WRITE TO BRONZE TABLE
# ---------------------------

(
    df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable(full_table_name)
)

display(df)